## Merge metadata files

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# --- CONFIGURATION ---
BASE_DIR = "/data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data"

# Original Paths
ORIG_MOL = f"{BASE_DIR}/mol_df.pkl"
ORIG_SPEC = f"{BASE_DIR}/spec_df.pkl"

# MoNA Paths
MONA_MOL = f"{BASE_DIR}/mol_df_mona.pkl"
MONA_SPEC = f"{BASE_DIR}/spec_df_mona.pkl"

# Output Paths (New Combined DBs)
OUT_MOL = f"{BASE_DIR}/mol_df_COMBINED.pkl"
OUT_SPEC = f"{BASE_DIR}/spec_df_COMBINED.pkl"

In [3]:
# --- 1. LOAD DATA ---
print("--- Loading Databases ---")
df_mol_orig = pd.read_pickle(ORIG_MOL)
df_spec_orig = pd.read_pickle(ORIG_SPEC)
print(f"Original: {len(df_mol_orig)} Mols, {len(df_spec_orig)} Specs")

df_mol_mona = pd.read_pickle(MONA_MOL)
df_spec_mona = pd.read_pickle(MONA_SPEC)
print(f"MoNA:     {len(df_mol_mona)} Mols, {len(df_spec_mona)} Specs")

--- Loading Databases ---
Original: 27969 Mols, 227308 Specs
MoNA:     6191 Mols, 33398 Specs


In [5]:
# --- 2. RESOLVE MOLECULE ID CONFLICTS ---
print("\n--- Resolving Mol ID Conflicts ---")

# Calculate the "Safe Offset"
# New IDs will start at (Max Original ID + 1)
max_id = df_mol_orig['mol_id'].max()
offset = max_id + 1
print(f" > Max Original Mol ID: {max_id}")
print(f" > Applying Offset of +{offset} to MoNA data...")

# Create copies to be safe
df_mol_mona_shifted = df_mol_mona.copy()
df_spec_mona_shifted = df_spec_mona.copy()


--- Resolving Mol ID Conflicts ---
 > Max Original Mol ID: 27968
 > Applying Offset of +27969 to MoNA data...


In [6]:
# A. Update Molecule DF
# Shift the ID itself
df_mol_mona_shifted['mol_id'] = df_mol_mona_shifted['mol_id'] + offset

# B. Update Spectrum DF
# Update the foreign key pointer to the molecule
df_spec_mona_shifted['mol_id'] = df_spec_mona_shifted['mol_id'] + offset

# Verify logic
print(f" > New MoNA Mol ID Range: {df_mol_mona_shifted['mol_id'].min()} - {df_mol_mona_shifted['mol_id'].max()}")

 > New MoNA Mol ID Range: 27969 - 34159


In [7]:
# --- 3. MERGE DATABASES ---
print("\n--- Merging DataFrames ---")

# Concatenate Molecules
# We use ignore_index=True to reset the dataframe row index (0..N), 
# but we keep the 'mol_id' column intact as the logical key.
df_mol_final = pd.concat([df_mol_orig, df_mol_mona_shifted], ignore_index=True)

# Concatenate Spectra
df_spec_final = pd.concat([df_spec_orig, df_spec_mona_shifted], ignore_index=True)


--- Merging DataFrames ---


In [9]:
# --- 4. DATA INTEGRITY CHECKS ---
print("\n--- Integrity Checks ---")

# Check 1: Duplicate Spec IDs?
# It is possible some MoNA spectra were already in your original set.
# If so, we drop the duplicates to avoid confusion.
if df_spec_final['spec_id'].duplicated().any():
    n_dupes = df_spec_final['spec_id'].duplicated().sum()
    print(f" > Warning: Found {n_dupes} duplicate Spec IDs. Keeping the original entries.")
    df_spec_final = df_spec_final.drop_duplicates(subset=['spec_id'], keep='first')

# Check 2: Orphaned Spectra?
# Do all spectra point to a valid molecule?
valid_mol_ids = set(df_mol_final['mol_id'])
orphans = df_spec_final[~df_spec_final['mol_id'].isin(valid_mol_ids)]

if len(orphans) > 0:
    print(f" > CRITICAL ERROR: Found {len(orphans)} orphaned spectra (mol_id not found in Mol DB).")
else:
    print(f" > Success: All {len(df_spec_final)} spectra link to valid molecules.")


--- Integrity Checks ---
 > Success: All 260706 spectra link to valid molecules.


In [10]:
# --- 5. SAVE ---
print(f"\n--- Saving Combined Databases ---")
df_mol_final.to_pickle(OUT_MOL)
df_spec_final.to_pickle(OUT_SPEC)

print(f"Saved to:")
print(f"  Mols: {OUT_MOL} ({len(df_mol_final)} records)")
print(f"  Specs: {OUT_SPEC} ({len(df_spec_final)} records)")


--- Saving Combined Databases ---
Saved to:
  Mols: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/mol_df_COMBINED.pkl (34160 records)
  Specs: /data/nas-gpu/wang/tmach007/SpectralSimilarityPredictor/mass_spec_gym_data/spec_df_COMBINED.pkl (260706 records)
